# 3c. Blueprint credential only: the telemetry path

Use this lesson if you want to send activity traces to **Agent 365 and Foundry / Application Insights** without configuring Graph or Blob tools.

**Path: [02 - Provision identity](02_provision_agent_identity.ipynb) -> 03c -> [04 - Registration and observability](04_agent365_registration_observability.ipynb).** This is an alternative to 03a/03b, not an extra requirement after them. If you already configured a blueprint credential there, reuse it and go straight to 04.

This notebook configures only the authentication proof for the **existing blueprint**. It does not create another identity, grant workload permissions, register the agent, acquire a telemetry token, or send telemetry. The Python cells only validate local configuration; the administrative commands below make real credential changes when you run them separately.

You will not configure Graph `User.Read`, OBO, `access_as_user`, a frontend sign-in client, Blob RBAC, or agent access to the Foundry model. Model access remains separate from telemetry.

## 0. Reuse the identities from notebook 02

Run from the repository directory using its `.venv` kernel and existing `requirements.txt` dependencies. Keep the following values in the ignored `.env` file:

```dotenv
ENTRA_TENANT_ID=YOUR-TENANT-GUID
ENTRA_BLUEPRINT_OBJECT_ID=YOUR-BLUEPRINT-OBJECT-GUID
ENTRA_BLUEPRINT_CLIENT_ID=YOUR-BLUEPRINT-APPLICATION-GUID
ENTRA_AGENT_IDENTITY_OBJECT_ID=YOUR-CHILD-AGENT-GUID
ENTRA_PROVISIONING_CLIENT_ID=YOUR-NOTEBOOK-02-PROVISIONING-APP-GUID
AGENT_IDENTITY_NAME=LearningAssistantLocal
```

The blueprint **object ID** identifies the directory object when managing credentials. Its **client ID** is used for runtime authentication. The child agent's object ID is also its client ID; it is not either blueprint ID. The provisioning app is an administrative tool, not the agent.

**No workload permissions does not mean no administrative setup.** An authorized owner/administrator must configure the blueprint credential. Notebook 04 separately requires consent on the provisioning app to register the child and read its existing sponsor. Do not assign those administrative Graph permissions to the runtime agent.

Choose **one** credential option below. Neither option needs an interactive user token at telemetry-export time.

## 1A. Local development: a short-lived blueprint secret

Reuse a valid blueprint secret if you already have one. Otherwise, an authorized blueprint owner/administrator can run this in a **separate PowerShell terminal**. Use PowerShell 7 with `Microsoft.Graph.Authentication` installed. This uses the Graph PowerShell client, not the notebook provisioning app. It requires `AgentIdentityBlueprint.AddRemoveCreds.All`, appropriate consent/user authority, and compliance with your organization's credential policies.

**Each run creates another secret.** This command does not print the secret; it copies the one-time value to your clipboard.

```powershell
Import-Module Microsoft.Graph.Authentication
$tenantId = 'YOUR-TENANT-GUID'
$blueprintObjectId = 'YOUR-BLUEPRINT-OBJECT-GUID'
Connect-MgGraph -TenantId $tenantId -Scopes 'AgentIdentityBlueprint.AddRemoveCreds.All' -ContextScope Process
$body = @{
    passwordCredential = @{
        displayName = 'Notebook03c-LocalDev'
        endDateTime = (Get-Date).ToUniversalTime().AddDays(1).ToString('yyyy-MM-ddTHH:mm:ssZ')
    }
} | ConvertTo-Json -Depth 4
$credential = Invoke-MgGraphRequest -Method POST `
    -Uri "https://graph.microsoft.com/v1.0/applications/$blueprintObjectId/microsoft.graph.agentIdentityBlueprint/addPassword" `
    -Body $body -ContentType 'application/json' -OutputType PSObject
Set-Clipboard -Value $credential.secretText
```

Paste the secret **value** into `.env`, enclosed in single quotes, and set:

```dotenv
A365_AUTH_MODE=client_secret
ENTRA_BLUEPRINT_CLIENT_SECRET='YOUR-SECRET-VALUE'
```

Save the file, then clear the clipboard with `Set-Clipboard -Value ''`. Do not print `$credential`, paste the secret into a notebook cell, or save it in outputs. Keep its `keyId` for cleanup; when finished, remove the development credential from the blueprint and the secret from `.env`. A compromised blueprint secret can affect its child identities, so prefer federation for production.

**Skip 1B and continue to section 2.**

## 1B. Azure hosting: managed identity and federation

Use an Azure host you control with a **user-assigned managed identity attached**. This notebook does not deploy the host or attach its identity. Local `az login` does not provide a managed identity endpoint.

Configure a federated identity credential (FIC) on the **existing blueprint** using supported Entra credential management. Reuse an existing matching trust rather than creating a duplicate. The equivalent Graph request is shown below for learning; this typed blueprint API is **preview**. Confirm support in your environment. It requires an authorized administrative Graph client, appropriate ownership/role, and consent; the current preview reference lists `AgentIdentityBlueprint.Create` or a higher blueprint permission. Notebook 02's permissions alone are not a guarantee of authorization.

```http
POST https://graph.microsoft.com/beta/applications/BLUEPRINT-OBJECT-ID/microsoft.graph.agentIdentityBlueprint/federatedIdentityCredentials
Content-Type: application/json
OData-Version: 4.0

{
  "name": "notebook03c-managed-identity",
  "issuer": "https://login.microsoftonline.com/TENANT-ID/v2.0",
  "subject": "MANAGED-IDENTITY-OBJECT-ID",
  "audiences": ["api://AzureADTokenExchange"]
}
```

The subject is the managed identity's **object/principal ID**, not its client ID. The tenant in the issuer must match your tenant exactly. On the Azure host, install the repository dependencies and configure its local `.env`:

```dotenv
A365_AUTH_MODE=managed_identity
AZURE_MANAGED_IDENTITY_CLIENT_ID=YOUR-MANAGED-IDENTITY-CLIENT-GUID
```

Keep the shared tenant/blueprint/child values from section 0. Do not copy a blueprint secret to the Azure host. The managed identity supplies proof to authenticate the blueprint; **telemetry is exported as the child agent identity**, not the host identity. No Blob or model role assignment is needed for this credential-only step.

Run notebook 04 on that Azure host too. Its explicit managed-identity mode does not fall back to your user sign-in or a secret.

## 2. Choose the telemetry destinations for notebook 04

Add these settings to `.env`:

```dotenv
A365_CAPTURE_CONTENT=false
A365_EXPORT_TO_FOUNDRY=true
APPLICATIONINSIGHTS_CONNECTION_STRING='YOUR-FOUNDRY-LINKED-APPLICATION-INSIGHTS-CONNECTION-STRING'
```

Replace the connection-string placeholder in your local `.env` with the value for the Application Insights resource linked to your Foundry project. Do not paste it into notebook code or outputs. Notebook 04 exports to **both destinations by default**; set `A365_EXPORT_TO_FOUNDRY=false` only if you want Agent 365 alone. The destination must allow the ingestion authentication used by the exporter; a connection string does not bypass Entra-only ingestion policies. Notebook 04 performs the export; this notebook does not test the destination.

`A365_CAPTURE_CONTENT=false` redacts manually recorded prompt, tool-result, and response content in 04. Agent 365 Activity consumes structured **activity spans**, not arbitrary `print()` or Python log lines. Foundry trace export and application-log collection are not the same thing.

Registered agent instances can export to Agent 365 with an **app-only token on the S2S route**, without an additional `Agent365.Observability.OtelWrite` permission or Observability admin consent, subject to service policy. This does not remove notebook 04's one-time administrative registration requirements. A standard, unregistered app registration is not equivalent to a registered agent instance.

## 3. Check local configuration (no network calls)

Save `.env` and restart the kernel first. Existing environment variables take precedence over `.env`. These cells check the required values without printing secrets, tokens, or IDs. They do **not** verify the secret, federated trust, identity relationships, registration, or ingestion access; notebook 04 makes the authenticated calls.

In [ ]:
import os
from uuid import UUID

from dotenv import load_dotenv

load_dotenv('.env')

def required(name):
    value = os.getenv(name, '').strip()
    if not value:
        raise ValueError(f'Set {name} in .env, then restart the kernel.')
    return value

def required_guid(name):
    try:
        value = UUID(required(name))
    except ValueError:
        raise ValueError(f'Set {name} to a valid nonzero GUID, then restart the kernel.') from None
    if value.int == 0:
        raise ValueError(f'Set {name} to a valid nonzero GUID, then restart the kernel.')
    return str(value)

tenant_id = required_guid('ENTRA_TENANT_ID')
blueprint_object_id = required_guid('ENTRA_BLUEPRINT_OBJECT_ID')
blueprint_client_id = required_guid('ENTRA_BLUEPRINT_CLIENT_ID')
agent_identity_id = required_guid('ENTRA_AGENT_IDENTITY_OBJECT_ID')
provisioning_client_id = required_guid('ENTRA_PROVISIONING_CLIENT_ID')
if len({blueprint_object_id, blueprint_client_id, agent_identity_id, provisioning_client_id}) != 4:
    raise ValueError('Use distinct blueprint object, blueprint client, child agent, and provisioning client IDs from notebook 02.')

auth_mode = required('A365_AUTH_MODE').lower()
if auth_mode == 'client_secret':
    if not os.getenv('ENTRA_BLUEPRINT_CLIENT_SECRET', '').strip():
        raise ValueError('Set ENTRA_BLUEPRINT_CLIENT_SECRET in .env; never paste it into a notebook cell.')
elif auth_mode == 'managed_identity':
    mi_client_id = required_guid('AZURE_MANAGED_IDENTITY_CLIENT_ID')
    if mi_client_id in {blueprint_client_id, agent_identity_id, provisioning_client_id}:
        raise ValueError('Use the Azure host managed identity client ID, not an agent or provisioning app ID.')
else:
    raise ValueError('A365_AUTH_MODE must be client_secret or managed_identity.')

for name, default in (('A365_CAPTURE_CONTENT', 'false'), ('A365_EXPORT_TO_FOUNDRY', 'true')):
    if os.getenv(name, default).lower() not in {'true', 'false'}:
        raise ValueError(f'{name} must be true or false.')
export_to_foundry = os.getenv('A365_EXPORT_TO_FOUNDRY', 'true').lower() == 'true'
if export_to_foundry and not os.getenv('APPLICATIONINSIGHTS_CONNECTION_STRING', '').strip():
    raise ValueError('Set APPLICATIONINSIGHTS_CONNECTION_STRING for the Foundry-linked Application Insights resource.')

print('Local configuration check passed; authentication has not been tested.')
print('Credential mode:', auth_mode)
print('Foundry trace export requested:', export_to_foundry)
print('Next: open notebook 04 with a fresh kernel.')

## 4. Continue to notebook 04

Open [04 - Register the agent and send activity traces](04_agent365_registration_observability.ipynb) with a **fresh kernel**. There is no Python object or token to carry over; only `.env` and your configured blueprint credential are reused.

1. Follow 04's prerequisites for the provisioning app: add/admin-consent delegated `AgentRegistration.ReadWrite.All` and `User.ReadBasic.All`, retaining `AgentIdentity.Read.All`. These administrative permissions belong on the provisioning app, not the runtime child. The child must already have the sponsor assigned during provisioning.
2. Keep the credential mode selected here. Run 04's registration step with `REGISTER` for first-time setup. It registers the existing child and caches its existing sponsor information.
3. Acquire the app-only Observability token and initialize the exporter on the **S2S route**. Notebook 04 handles both secret and managed-identity modes.
4. Run its smoke-test activity, which needs no model or user token at export time. Optionally trace a real model call and verify delivery in both destinations if Foundry export is enabled.

You can skip 04's optional model call and Foundry External Agent registration. Sending traces to the Foundry-linked Application Insights resource does not itself require creating a Foundry External Agent. If you do run a model call, its model-access permissions are independent of this telemetry credential.

Keep `.env` and local registration state out of shared folders and screenshots. Do not delete registration state to retry an ambiguous registration failure. An accepted telemetry export alone is not proof that Activity is visible; complete 04's verification steps.

### References

- [Microsoft OpenTelemetry distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry?tabs=python)
- [Registered-agent S2S observability authentication](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/observability-authentication-setup#agent-365-enabled-using-s2s)
- [Add a blueprint password](https://learn.microsoft.com/en-us/graph/api/agentidentityblueprint-addpassword?view=graph-rest-1.0)
- [Create a blueprint federated identity credential](https://learn.microsoft.com/en-us/graph/api/agentidentityblueprint-post-federatedidentitycredentials?view=graph-rest-beta)